# Colab runner — Group 2 (Usman's DeBERTa pipeline)

1. Runtime → Change runtime type → **T4 GPU**
2. Put `train.csv`, `unlabeled.csv`, `test.csv` (+ `sample_submission.csv`) anywhere in your Google Drive (e.g. `MyDrive/aml-human-or-ai/data/`).
3. Runtime → **Run all**. Results are saved to `MyDrive/aml-human-or-ai/results/`, so a disconnect just means Run all again (finished steps are skipped).
4. At the end a `results.zip` downloads: unzip it into `submissions/group-2-jerry-varun-usman/results/` in the repo.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import glob, os, pandas as pd
# pick the folder that has the competition files (not some other train.csv)
cands = [os.path.dirname(p) for p in glob.glob('/content/drive/MyDrive/**/train.csv', recursive=True)]
cands = [d for d in cands
         if all(os.path.exists(os.path.join(d, f)) for f in ('unlabeled.csv', 'test.csv'))
         and 'text' in pd.read_csv(os.path.join(d, 'train.csv'), nrows=1).columns]
assert cands, 'No folder in Drive has train.csv (with a text column) + unlabeled.csv + test.csv'
DATA_DIR = cands[0]
RESULTS_DIR = '/content/drive/MyDrive/aml-human-or-ai/results'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.environ['AML_DATA_DIR'] = DATA_DIR
os.environ['AML_RESULTS_DIR'] = RESULTS_DIR
print('data:', DATA_DIR, os.listdir(DATA_DIR))

In [ ]:
%cd /content
REPO = 'https://github.com/usmanali-arshi/human-or-ai-submissions-group2.git'
BRANCH = 'usman/deberta-pipeline'
!rm -rf /content/repo && git clone -q -b {BRANCH} {REPO} /content/repo
%cd /content/repo/submissions/group-2-jerry-varun-usman
!git log --oneline -3
!pip -q install -r requirements.txt

In [ ]:
# Q1 classical baselines (CPU, ~5-10 min)
!python baselines.py

In [ ]:
# DeBERTa: main -> learning curve -> pseudo-labels -> final (GPU, ~1 hour total)
!python train_deberta.py --mode all

In [ ]:
# Tables, plot, error analysis, and the Kaggle submission
!python analyze.py
!python predict.py --out "$AML_RESULTS_DIR/submission.csv"

In [ ]:
# Q5 / Kaggle: DeBERTa-v3-base, 512 tokens (~40-50 min on a T4). Cached like the rest.
FLAGS = "--tag base512 --model-name microsoft/deberta-v3-base --max-len 512 --batch-size 8 --grad-accum 2"
!python train_deberta.py --mode main {FLAGS}
!python train_deberta.py --mode final {FLAGS}
!python blend.py --submit base512 deberta --out "$AML_RESULTS_DIR/submission_blend.csv"
!python blend.py --submit base512 --out "$AML_RESULTS_DIR/submission_base512.csv"

In [ ]:
!cd /content/drive/MyDrive/aml-human-or-ai && zip -qr /content/results.zip results
from google.colab import files; files.download('/content/results.zip')